In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from timeit import default_timer as timer

from mpl_toolkits.mplot3d import Axes3D

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble.partial_dependence import plot_partial_dependence
from sklearn.ensemble.partial_dependence import partial_dependence
from sklearn.datasets import load_boston

import lightgbm as lgb

from mli.explanation.h_statistic import HStatistic
from mli.explanation.pd import PD


botons_housing = load_boston()
print(botons_housing.DESCR)

# split 80/20 train-test
X_train, X_test, y_train, y_test = train_test_split(botons_housing.data,
                                                        botons_housing.target,
                                                        test_size=0.2,
                                                        random_state=1)

Boston House Prices dataset

Notes
------
Data Set Characteristics:  

    :Number of Instances: 506 

    :Number of Attributes: 13 numeric/categorical predictive
    
    :Median Value (attribute 14) is usually the target

    :Attribute Information (in order):
        - CRIM     per capita crime rate by town
        - ZN       proportion of residential land zoned for lots over 25,000 sq.ft.
        - INDUS    proportion of non-retail business acres per town
        - CHAS     Charles River dummy variable (= 1 if tract bounds river; 0 otherwise)
        - NOX      nitric oxides concentration (parts per 10 million)
        - RM       average number of rooms per dwelling
        - AGE      proportion of owner-occupied units built prior to 1940
        - DIS      weighted distances to five Boston employment centres
        - RAD      index of accessibility to radial highways
        - TAX      full-value property-tax rate per $10,000
        - PTRATIO  pupil-teacher ratio by town
      

In [12]:
feature_name = list(botons_housing['feature_names'])
print(f"Feature Names: {feature_name}")

Feature Names: ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']


In [3]:
# Since, the loaded data is in the `sklearn.utils.Bunch` format we have to manually convert it to pandas dataframe

data_df_train = pd.DataFrame(data=X_train, columns=feature_name)

data_df_test = pd.DataFrame(data=X_test, columns=feature_name)
print(data_df_train.head(5))

print(f"Shape of the training data {data_df_train.shape}")
print(f"Shape of the test data {data_df_test.shape}")

       CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS   RAD    TAX  \
0   0.14150   0.0   6.91   0.0  0.448  6.169   6.6  5.7209   3.0  233.0   
1   0.15445  25.0   5.13   0.0  0.453  6.145  29.2  7.8148   8.0  284.0   
2  16.81180   0.0  18.10   0.0  0.700  5.277  98.1  1.4261  24.0  666.0   
3   0.05646   0.0  12.83   0.0  0.437  6.232  53.7  5.0141   5.0  398.0   
4   8.79212   0.0  18.10   0.0  0.584  5.565  70.6  2.0635  24.0  666.0   

   PTRATIO       B  LSTAT  
0     17.9  383.37   5.81  
1     19.7  390.68   6.86  
2     20.2  396.90  30.81  
3     18.7  386.40  12.34  
4     20.2    3.65  17.16  
Shape of the training data (404, 13)
Shape of the test data (102, 13)


In [13]:
# First just using lightgbm
lgb_train = lgb.Dataset(X_train, y_train, free_raw_data=False)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train, free_raw_data=False)

# specify your configurations as a dict
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'regression_l2', 
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

In [14]:
print(feature_name)

['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']


In [15]:
print('Starting training...')
# feature_name and categorical_feature
gbm = lgb.train(params,
                lgb_train,
                num_boost_round=40,
                valid_sets=lgb_train,  # eval training data
                feature_name=feature_name)

Starting training...
[1]	training's l2: 74.6485
[2]	training's l2: 68.9987
[3]	training's l2: 63.7749
[4]	training's l2: 59.0347
[5]	training's l2: 55.2248
[6]	training's l2: 51.5263
[7]	training's l2: 48.1278
[8]	training's l2: 45.1453
[9]	training's l2: 42.406
[10]	training's l2: 39.7968
[11]	training's l2: 37.5691
[12]	training's l2: 35.387
[13]	training's l2: 33.1515
[14]	training's l2: 31.1484
[15]	training's l2: 29.456
[16]	training's l2: 28.0863
[17]	training's l2: 26.6276
[18]	training's l2: 25.4232
[19]	training's l2: 24.3541
[20]	training's l2: 23.2506
[21]	training's l2: 22.2107
[22]	training's l2: 21.2613
[23]	training's l2: 20.4704
[24]	training's l2: 19.615
[25]	training's l2: 18.8224
[26]	training's l2: 18.0581
[27]	training's l2: 17.3707
[28]	training's l2: 16.754
[29]	training's l2: 16.1425
[30]	training's l2: 15.5918
[31]	training's l2: 15.1047
[32]	training's l2: 14.5941
[33]	training's l2: 14.1221
[34]	training's l2: 13.749
[35]	training's l2: 13.3725
[36]	training'

## Quick Evaluation

In [18]:
def compute_metrics(model_instance, input_X , input_y):
    # simply computing the response on dependent data
    y_hat = model_instance.predict(input_X)
    
    # Compute different metrics
    sum_sqaure_error = np.sum(np.square(input_y - y_hat))
    sum_of_square_total = np.sum(np.square(input_y - np.mean(input_y)))
    
    # Computing R^2
    r_2 = 1 - (float(sum_sqaure_error))/sum_of_square_total
    
    # Computing adjusted R^2
    adj_r_2 = 1 - ((1-r_2)*(len(input_y)-1))/(len(input_y)- input_X.shape[1] - 1)
    return ({"R2":r_2, "Adjusted R2":adj_r_2})

In [19]:
# Test
result = compute_metrics(gbm, X_test, y_test)
print(f"Model Diagnostics on Test Data: {result}")

Model Diagnostics on Test Data: {'R2': 0.8229875018149405, 'Adjusted R2': 0.7968379282194203}


In [20]:
max_val = np.max(gbm.feature_importance())
relative_importance = [float(v)/max_val for v in gbm.feature_importance()]
print(relative_importance)

#feature_rank = pd.DataFrame(np.array(relative_importance), columns=feature_name)

[0.44086021505376344, 0.021505376344086023, 0.13978494623655913, 0.0, 0.3225806451612903, 0.8602150537634409, 0.5053763440860215, 0.4838709677419355, 0.053763440860215055, 0.34408602150537637, 0.4946236559139785, 0.3010752688172043, 1.0]


In [21]:
feature_rank = pd.DataFrame(relative_importance).T
feature_rank.columns = feature_name

result = feature_rank.T.sort_values([0], ascending=False)
print(result)

                0
LSTAT    1.000000
RM       0.860215
AGE      0.505376
PTRATIO  0.494624
DIS      0.483871
CRIM     0.440860
TAX      0.344086
NOX      0.322581
B        0.301075
INDUS    0.139785
RAD      0.053763
ZN       0.021505
CHAS     0.000000


## Computing Interactions

In [22]:
def predict(x):
    # Multi-class use-case
    return pd.DataFrame(gbm.predict(x))

# sanity check
predict(data_df_train.loc[0])

/home/ubuntu/anaconda3/envs/mli_dev/lib/python3.6/site-packages/lightgbm/basic.py:469: UserWarning: Converting data to scipy sparse matrix.
  warnings.warn('Converting data to scipy sparse matrix.')


,0
0,24.826229


In [23]:
X_train_df = pd.DataFrame(X_train, columns=feature_name)
interactions = HStatistic("H-stat for pairs").explain(
            feature_name,
            data_df_train,
            predict_method=predict)

In [24]:
interactions.explanations()

{('CRIM', 'ZN'): {'p_0': 0.00016992481362463447},
 ('CRIM', 'INDUS'): {'p_0': 0.011994604971967587},
 ('CRIM', 'CHAS'): {'p_0': 2.661221666446386e-27},
 ('CRIM', 'NOX'): {'p_0': 0.24096850463100422},
 ('CRIM', 'RM'): {'p_0': 0.0010108002523923074},
 ('CRIM', 'AGE'): {'p_0': 0.047014169726963505},
 ('CRIM', 'DIS'): {'p_0': 0.032639677306517796},
 ('CRIM', 'RAD'): {'p_0': 3.4635169343432866e-27},
 ('CRIM', 'TAX'): {'p_0': 0.029616890463141164},
 ('CRIM', 'PTRATIO'): {'p_0': 0.06934403293564625},
 ('CRIM', 'B'): {'p_0': 0.003806000022782893},
 ('CRIM', 'LSTAT'): {'p_0': 0.018955455741480814},
 ('ZN', 'INDUS'): {'p_0': 7.72901902615522e-28},
 ('ZN', 'CHAS'): {'p_0': 1.6216024813170959e-25},
 ('ZN', 'NOX'): {'p_0': 6.117935173010185e-29},
 ('ZN', 'RM'): {'p_0': 9.761133035616682e-05},
 ('ZN', 'AGE'): {'p_0': 0.00039511673898237295},
 ('ZN', 'DIS'): {'p_0': 9.18147785520909e-28},
 ('ZN', 'RAD'): {'p_0': 3.559400616002283e-27},
 ('ZN', 'TAX'): {'p_0': 7.125489246324884e-05},
 ('ZN', 'PTRATIO'